In [0]:
from pyspark.sql.types import *

employee_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("department_id", IntegerType(), True)
])
employee_data = [
    (10306, "Ashley", "Li", 28516, 4),
    (10307, "Joseph", "Solomon", 19945, 1),
    (10311, "Melissa", "Holmes", 33575, 1),
    (10316, "Beth", "Torres", 34902, 1),
    (10317, "Pamela", "Rodriguez", 48187, 4),
    (10320, "Gregory", "Cook", 22681, 4),
    (10324, "William", "Brewer", 15947, 1),
    (10329, "Christopher", "Ramos", 37710, 4),
    (10333, "Jennifer", "Blankenship", 13433, 4),
    (10339, "Robert", "Mills", 13188, 1)
]

employee_df = spark.createDataFrame(employee_data, employee_schema)
employee_df.show(truncate=False)

dept_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("department", StringType(), True)
])

dept_data = [
    (1, "engineering"),
    (2, "human resource"),
    (3, "operation"),
    (4, "marketing")
]

dept_df = spark.createDataFrame(dept_data, dept_schema)
from pyspark.sql.functions import col,max,lead
from pyspark.sql.window import Window
employee_dept_df=employee_df.alias("t1").join(dept_df.alias("t2"),on=col("t1.department_id")==col("t2.id"),how="inner").filter(col("department").isin("marketing","engineering"))

df=(employee_dept_df.groupBy(
    "department"
).agg(
    max(col("salary")).alias("max_salary")
))
window_spec = Window.orderBy(col("max_salary"))
df = df.withColumn(
    "next_sal",
    lead(col("max_salary")).over(window_spec)
)

display(df.withColumn("sal_dif",col("max_salary")-col("next_sal")))
